In [2]:
import numpy as np
import asf_search
import datetime
asf_search.constants.INTERNAL.CMR_TIMEOUT = 120
#session = asf_search.ASFSession().auth_with_creds('your_earthdata_username', 'your_earthdata_password')

In [3]:
help(asf_search.constants)

Help on package asf_search.constants in asf_search:

NAME
    asf_search.constants

DESCRIPTION
    Various constants to be used in search and related functions,
    provided as a convenience to help ensure sensible values.

PACKAGE CONTENTS
    BEAMMODE
    DATASET
    FLIGHT_DIRECTION
    INSTRUMENT
    INTERNAL
    PLATFORM
    POLARIZATION
    PRODUCTION_CONFIGURATION
    PRODUCT_TYPE
    RANGE_BANDWIDTH

DATA
    AIRSAR = 'AIRSAR'
    ALOS = 'ALOS'
    ALOS_2 = 'ALOS-2'
    ALOS_AVNIR_2 = 'ALOS AVNIR-2'
    ALOS_PALSAR = 'ALOS PALSAR'
    AMPLITUDE = 'AMPLITUDE'
    AMPLITUDE_GRD = 'AMPLITUDE_GRD'
    ARIA_S1_GUNW = 'ARIA S1 GUNW'
    ASCENDING = 'ASCENDING'
    ASF_AUTH_HOST = 'auth.asf.alaska.edu'
    AUTH_COOKIES = ['urs_user_already_logged', 'uat_urs_user_already_logge...
    AUTH_DOMAINS = ['asf.alaska.edu', 'earthdata.nasa.gov']
    AVNIR_2 = 'AVNIR-2'
    BURST = 'BURST'
    BW_10 = '10'
    BW_20 = '20'
    BW_20_5 = '20+5'
    BW_25 = '25'
    BW_37 = '37'
    BW_40 = '40

In [20]:
# validation test
#results = asf_search.granule_search(['ALPSRS279162400', 'ALPSRS279162200'])
#print(results)

{
  "features": [
    {
      "geometry": {
        "coordinates": [
          [
            [
              -144.207,
              61.899
            ],
            [
              -145.235,
              58.851
            ],
            [
              -139.288,
              58.195
            ],
            [
              -137.7,
              61.211
            ],
            [
              -144.207,
              61.899
            ]
          ]
        ],
        "type": "Polygon"
      },
      "properties": {
        "beamModeType": "WB1",
        "browse": [
          "https://datapool.asf.alaska.edu/BROWSE/A3/ALPSRS279162400.jpg"
        ],
        "bytes": 23900540,
        "centerLat": 60.077,
        "centerLon": -141.6007,
        "faradayRotation": 5.605801,
        "fileID": "ALPSRS279162400-L1.5",
        "fileName": "ALPSRS279162400-L1.5.zip",
        "flightDirection": "DESCENDING",
        "frameNumber": 2400,
        "granuleType": "ALOS_PALSAR_SCENE",
       

In [4]:
#searchPoly = "POLYGON ((-150 60,-150 80,-70 80,-70 60,-150 60))"
searchPoly = "POLYGON((-136.6025 64.3116,-71.7744 64.3116,-71.7744 79.3352,-136.6025 79.3352,-136.6025 64.3116))"

results = asf_search.geo_search(
    intersectsWith = searchPoly,
    dataset = 'ALOS-2',
    polarization = 'quadrature',
    flightDirection='ASCENDING',
    start='2007-10-30T11:59:59Z',
    end='2007-10-30T11:59:59Z'
    season = [120, 150]
)

In [3]:
import pickle
#with open("asf_search_cache.pickle", 'wb') as file:
#    pickle.dump(results, file)
with open("asf_search_cache.pickle", "rb") as file:
    results = pickle.load(file)

In [5]:
results[0].

In [7]:
import matplotlib.pyplot as plt
khorn = np.array(results[0].geometry['coordinates'][0])
cen = khorn.mean(axis=0)
khorn = (khorn[:,0].max() - khorn[:,0].min(), khorn[:,1].max() - khorn[:,1].min())
print (khorn, cen)
#print(bound, cen)
#judg = bound - cen
#judg

#plt.scatter(bound[:,0], bound[:,1])

(np.float64(1.1800000000000068), np.float64(0.6349999999999909)) [-133.827    71.3888]


In [30]:
r1 = np.array(results[0].geometry['coordinates'][0][:-1])
r2 = np.array(results[10].geometry['coordinates'][0][:-1])
thresh = np.array((r2[:,0].max() - r2[:,0].min(), r2[:,1].max() - r2[:,1].min()))

print((r2.mean(axis=0) - r1.mean(axis=0)))
print(thresh)
#print(r1, '\n\n', r2)
np.sum((r2.mean(axis=0) - r1.mean(axis=0))**2) * 1.5 < np.sum(thresh**2)

[ 1.42975 -1.42025]
[1.08  0.634]


np.False_

In [44]:
import numpy as np
filtered_results = [res for res in results if ('-L1.1' in res.meta['native-id'])]
results_with_overlap = []
#already_true = [False for i in range(len(results))]
for res1 in filtered_results:
    r1 = np.array(res1.geometry['coordinates'][0][:-1])
    thresh = np.array((r1[:,0].max() - r1[:,0].min(), r1[:,1].max() - r1[:,1].min())) * 0.5
    for res2 in filtered_results:
        r2 = np.array(res2.geometry['coordinates'][0][:-1])
        rd = np.abs(r2.mean(axis=0) - r1.mean(axis=0))
        if (np.sum(rd) == 0): continue
        if (rd[0] * 2 < thresh[0] and rd[1] * 2 < thresh[1]):
            results_with_overlap.append(res1)
            break
        #else: print('failed!')
results_with_overlap = list(set(results_with_overlap))
#results[0].geometry['coordinates'][0]

In [45]:
len(results_with_overlap)

49

In [47]:
#(results_with_overlap[0].get_urls())
strcsv = ""
for rfile in results_with_overlap:
    strcsv += rfile.meta['native-id'].split('-')[0] + ', '
strcsv

'ALPSRP174611580, ALPSRP176361530, ALPSRP174611570, ALPSRP176361520, ALPSRP174771600, ALPSRP176361510, ALPSRP177991400, ALPSRP177251610, ALPSRP174611560, ALPSRP174611550, ALPSRP176361540, ALPSRP177991350, ALPSRP174611540, ALPSRP174611530, ALPSRP121831350, ALPSRP174611520, ALPSRP121831370, ALPSRP177991380, ALPSRP174771590, ALPSRP174611510, ALPSRP177251600, ALPSRP121831390, ALPSRP176361620, ALPSRP121831410, ALPSRP176361610, ALPSRP176361500, ALPSRP176361600, ALPSRP176361590, ALPSRP177991360, ALPSRP174611620, ALPSRP176361580, ALPSRP176361570, ALPSRP174611610, ALPSRP177251590, ALPSRP176361490, ALPSRP174611600, ALPSRP121831400, ALPSRP176361560, ALPSRP174611590, ALPSRP177991340, ALPSRP174611500, ALPSRP174771610, ALPSRP121831360, ALPSRP176361550, ALPSRP177991390, ALPSRP121831340, ALPSRP177991410, ALPSRP121831380, ALPSRP177991370, '

In [53]:
dir(results_with_overlap[0])

['BaselineCalcType',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_base_properties',
 '_get_access_urls',
 '_get_additional_filenames_and_urls',
 '_get_additional_urls',
 '_get_s3_uris',
 '_get_urls',
 '_is_subclass',
 '_parse_filename_from_url',
 '_read_property',
 '_read_umm_property',
 '_url_types',
 'baseline',
 'baseline_type',
 'centroid',
 'download',
 'find_urls',
 'geojson',
 'geometry',
 'get_baseline_calc_properties',
 'get_classname',
 'get_default_baseline_product_type',
 'get_sort_keys',
 'get_stack_opts',
 'get_urls',
 'has_baseline',
 'is_valid_reference',
 'meta',
 'properties',
 'remotezip',
 'session',
 'stack',
 'translate_produ

In [74]:
results_with_overlap[25].umm

{'TemporalExtent': {'RangeDateTime': {'BeginningDateTime': '2009-05-17T02:29:09.000Z',
   'EndingDateTime': '2009-05-17T02:29:17.000Z'}},
 'OrbitCalculatedSpatialDomains': [{'OrbitNumber': 17636}],
 'GranuleUR': 'ALPSRP176361500-L1.1',
 'AdditionalAttributes': [{'Name': 'FLIGHT_LINE', 'Values': ['NULL']},
  {'Name': 'GROUP_ID', 'Values': ['ALPSRP176361500']},
  {'Name': 'OFF_NADIR_ANGLE', 'Values': ['21.5']},
  {'Name': 'MD5SUM', 'Values': ['77ef75520cf189c8e91132a55501c0a1']},
  {'Name': 'GRANULE_TYPE', 'Values': ['ALOS_PALSAR_SCENE']},
  {'Name': 'ASCENDING_DESCENDING', 'Values': ['ASCENDING']},
  {'Name': 'FAR_END_LAT', 'Values': ['74.469']},
  {'Name': 'INSAR_STACK_SIZE', 'Values': ['2']},
  {'Name': 'BEAM_MODE_TYPE', 'Values': ['PLR']},
  {'Name': 'INSAR_BASELINE', 'Values': ['0']},
  {'Name': 'CENTER_FRAME_ID', 'Values': ['1483']},
  {'Name': 'CENTER_ESA_FRAME', 'Values': ['1483']},
  {'Name': 'ACQUISITION_DATE', 'Values': ['2009-05-17T02:29:17Z']},
  {'Name': 'MISSION_NAME', 'Va